IMPORTS

In [1]:
from typing import List, Dict, Any, Set
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import os
import json
from datasets import load_dataset
from tqdm import tqdm
import pandas as pd
from rouge_score import rouge_scorer
import sacrebleu

/home/estudiante/tldr-uniandes/encoders-vs-decoders-classification/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Script de Evaluación de Modelos de Lenguaje (Sin Fine-Tuning).

Este script carga un modelo de lenguaje pre-entrenado (Gemma-3) en modo 4 bits,
realiza inferencia sobre un dataset de test y calcula métricas de calidad
(ROUGE-2, ROUGE-L y BLEU) comparando los resúmenes generados con los de referencia.

Funcionalidades principales:
1. Carga de modelo optimizada (BitsAndBytes 4-bit).
2. Gestión de estado (resume) para continuar ejecuciones interrumpidas.
3. Inferencia con truncamiento inteligente de contexto.
4. Cálculo y reporte de métricas de evaluación.

CONFIGURACIÓN

In [2]:
HF_DATASET_NAME: str = "andrewmos/indian-legal-summaries-chat-template"
MODEL_NAME: str = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit"
MAX_NEW_TOKENS: int = 1024
MAX_INPUT_TOKENS: int = 8000  # Margen de seguridad (Contexto total ~8192)
jsonl_file: str = "summaries_sin_finetuning.jsonl"

CARGA DEL MODELO (ESTÁNDAR HUGGING FACE)

In [3]:
print("Cargando modelo con Transformers (Modo Estable)...")

# Configuración para cargar en 4 bits (ahorra memoria igual que Unsloth)
bnb_config: BitsAndBytesConfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

try:
    tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model: AutoModelForCausalLM = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",  # Usa la GPU automáticamente
        torch_dtype=torch.float16,
        attn_implementation="sdpa" # Usa Flash Attention si está disponible (más rápido)
    )
    print("Modelo cargado correctamente en GPU.")
except Exception as e:
    print(f"Error cargando modelo: {e}")
    exit()

Cargando modelo con Transformers (Modo Estable)...


`torch_dtype` is deprecated! Use `dtype` instead!


Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


/home/estudiante/tldr-uniandes/encoders-vs-decoders-classification/venv/lib/python3.10/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Modelo cargado correctamente en GPU.


CARGAR DATASET

In [4]:
dataset_eval = load_dataset(HF_DATASET_NAME, split="test")
print(f"Total test samples originales: {len(dataset_eval)}")

Total test samples originales: 240


GESTIÓN DE CONTINUACIÓN (RESUME)

In [5]:
summary_store: Dict[str, str] = {}
if os.path.exists(jsonl_file):
    print(f"📂 Archivo encontrado: {jsonl_file}. Cargando registros previos...")
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    data: Dict[str, Any] = json.loads(line)
                    _id: str = data.get("ID", "")
                    summary_store[_id] = str(data.get("Summary", "")).strip()
                except:
                    continue
    print(f"   ✔️ {len(summary_store)} registros cargados")
else:
    print("📝 Archivo nuevo, no hay registros previos.")

# Filtrar ya procesados y vacíos
ids_ok: Set[str] = {k for k, v in summary_store.items() if v and v.lower() not in ["null", "none", ""]}
dataset_eval = dataset_eval.filter(lambda x: (x["ID"] not in ids_ok))
print(f"➡️ Total que se procesarán ahora: {len(dataset_eval)}")

📂 Archivo encontrado: summaries_sin_finetuning.jsonl. Cargando registros previos...
   ✔️ 240 registros cargados
➡️ Total que se procesarán ahora: 0


LOOP DE INFERENCIA

In [6]:
generated_summaries: List[Dict[str, str]] = []

print(f"Iniciando inferencia en {len(dataset_eval)} muestras...")

for i, row in enumerate(tqdm(dataset_eval)):
    row_id: str = row["ID"]
    row_input: str = row["messages"][0]["content"]

    # A. Preparar mensaje
    messages: List[Dict[str, str]] = [{"role": "user", "content": row_input}]

    # B. Tokenizar
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # C. Truncamiento inteligente
    # Corta solo si excede el límite, manteniendo el inicio (instrucción)
    input_len: int = inputs["input_ids"].shape[-1]
    if input_len > MAX_INPUT_TOKENS:
        inputs["input_ids"] = inputs["input_ids"][:, :MAX_INPUT_TOKENS]
        inputs["attention_mask"] = inputs["attention_mask"][:, :MAX_INPUT_TOKENS]

    # D. Generar
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False  # Greedy decoding para evaluación
        )

    # E. Decodificar (corta el prompt de entrada)
    # Calcula la longitud real del input usado (puede haber sido truncado)
    len_input_real: int = inputs["input_ids"].shape[-1]
    prediction: str = tokenizer.decode(outputs[0][len_input_real:], skip_special_tokens=True).strip()

    generated_summaries.append({"ID": row_id, "Summary": prediction})

    # Guardado parcial cada 10 items
    if (i + 1) % 10 == 0:
        with open(jsonl_file, "a", encoding="utf-8") as f:
            for item in generated_summaries:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        # Actualizar store y limpiar buffer
        summary_store.update({x["ID"]: x["Summary"] for x in generated_summaries})
        generated_summaries = []

# Guardar remanentes
if generated_summaries:
    with open(jsonl_file, "a", encoding="utf-8") as f:
        for item in generated_summaries:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    summary_store.update({x["ID"]: x["Summary"] for x in generated_summaries})

Iniciando inferencia en 0 muestras...


0it [00:00, ?it/s]

0it [00:00, ?it/s]

LIMPIEZA FINAL Y MÉTRICAS

In [7]:
print("\nReescribiendo archivo final limpio...")
with open(jsonl_file, "w", encoding="utf-8") as f:
    for _id, summary in summary_store.items():
        line = json.dumps({"ID": _id, "Summary": summary}, ensure_ascii=False)
        f.write(line + "\n")

print("\nCALCULANDO MÉTRICAS GLOBALES...")
scorer = rouge_scorer.RougeScorer(["rouge2", "rougeL"], use_stemmer=True)
all_metrics_global: List[Dict[str, Any]] = []

# Recargar dataset completo para comparar
full_dataset = load_dataset(HF_DATASET_NAME, split="test")

for row in full_dataset:
    row_id = row["ID"]
    if row_id in summary_store:
        pred: str = summary_store[row_id]
        ref: str = row["messages"][1]["content"]

        rouge_scores = scorer.score(ref, pred)
        rouge2: float = rouge_scores["rouge2"].fmeasure
        rougel: float = rouge_scores["rougeL"].fmeasure
        
        # Sacrebleu
        bleu: float = sacrebleu.corpus_bleu([pred], [[ref]]).score / 100
        avg: float = (rouge2 + rougel + bleu) / 3

        all_metrics_global.append({
            "id": row_id, "rouge2": rouge2, "rougeL": rougel, "bleu": bleu, "avg": avg
        })

if all_metrics_global:
    df_global = pd.DataFrame(all_metrics_global)
    print(df_global.describe())
    print(f"\nResultados Finales:")
    print(f"  R2: {df_global['rouge2'].mean():.4f} | RL: {df_global['rougeL'].mean():.4f} | BLEU: {df_global['bleu'].mean():.4f}")
    print(f"  AVG: {df_global['avg'].mean():.4f}")
else:
    print("No hay métricas disponibles (dataset vacío o errores).")


Reescribiendo archivo final limpio...

CALCULANDO MÉTRICAS GLOBALES...


           rouge2      rougeL        bleu         avg
count  240.000000  240.000000  240.000000  240.000000
mean     0.117947    0.179810    0.045146    0.114301
std      0.077871    0.072856    0.057812    0.065874
min      0.000000    0.001969    0.000000    0.001248
25%      0.072367    0.150613    0.007158    0.077195
50%      0.108334    0.188124    0.025621    0.108185
75%      0.150324    0.212432    0.059222    0.142689
max      0.465224    0.403071    0.374356    0.382226

Resultados Finales:
  R2: 0.1179 | RL: 0.1798 | BLEU: 0.0451
  AVG: 0.1143
